# 🌳 Decision Trees - In Depth

Decision Trees are the most interpretable algorithms in Machine Learning. They are called "White Box" models because you can physically print out the exact logical steps the AI took to reach a conclusion.

If you need to explain to a CEO *why* the AI denied a customer's loan, you use a Decision Tree.

## 🧠 1. Deep Dive into the Theory

A Decision Tree splits data based on questions. But how does it pick the questions? It looks for the feature that creates the most **Pure** subgroups.

If we have a bucket of 100 people (50 bought an item, 50 didn't):
- Split 1: "Are they over 30?" -> Results in a group of 30 buyers and 20 non-buyers (Messy).
- Split 2: "Do they make over $50k?" -> Results in a group of 45 buyers and 5 non-buyers (Very Pure!).

The tree will always choose Split 2 because it reduces the "chaos" or "impurity" of the data the most.

## 🧮 2. The Math: Gini Impurity by Hand

Scikit-Learn uses **Gini Impurity** by default.
$$ Gini = 1 - \sum (p_i)^2 $$
Where $p_i$ is the probability of an item belonging to class $i$.

**Example:**
Imagine a node with 3 Apples (Class A) and 1 Orange (Class B).
- Probability of Apple ($p_A$) = $3/4 = 0.75$
- Probability of Orange ($p_B$) = $1/4 = 0.25$

$$ Gini = 1 - (0.75^2 + 0.25^2) = 1 - (0.5625 + 0.0625) = 1 - 0.625 = 0.375 $$

If the node was perfectly pure (4 Apples, 0 Oranges):
$$ Gini = 1 - (1^2 + 0^2) = 0 $$
The algorithm actively searches for splits that drive the Gini Impurity down to 0!

## 💻 3. Implementation and Hyperparameter Tuning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load a real-world dataset: Breast Cancer Wisconsin
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dataset Shape:", X.shape)

### The Overfitting Problem
If we don't tell the tree to stop, it will grow until every single leaf is 100% pure, effectively memorizing the training data. Let's see what an overfitted tree looks like.

In [ ]:
# Train an unrestricted tree
unrestricted_tree = DecisionTreeClassifier(random_state=42)
unrestricted_tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, unrestricted_tree.predict(X_train))
test_acc = accuracy_score(y_test, unrestricted_tree.predict(X_test))

print(f"Training Accuracy (Unrestricted): {train_acc*100:.2f}% (Perfect Memorization!)")
print(f"Testing Accuracy (Unrestricted): {test_acc*100:.2f}%")

## 🛡️ 4. Taming the Tree with GridSearch

To stop overfitting, we use **Pruning**. We restrict:
- `max_depth`: How deep the tree can go.
- `min_samples_split`: Minimum samples required to split an internal node.
- `min_samples_leaf`: Minimum samples required to be at a leaf node.

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

best_tree = grid_search.best_estimator_
print("Best Hyperparameters:", grid_search.best_params_)
print(f"New Testing Accuracy: {accuracy_score(y_test, best_tree.predict(X_test))*100:.2f}%")

## 🧐 5. Extracting Feature Importance and Visualization

Decision trees automatically rank features by how much they decrease impurity. This is incredibly valuable for business insights!

In [ ]:
importances = pd.Series(best_tree.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(10).plot(kind='barh', color='darkgreen').invert_yaxis()
plt.title('Top 10 Most Important Features for Predicting Breast Cancer')
plt.xlabel('Importance (Gini Reduction)')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(best_tree, filled=True, feature_names=X.columns, class_names=data.target_names, rounded=True, max_depth=2)
plt.title("The Top Branches of Our Pruned Decision Tree")
plt.show()

## 📊 6. Summary: Pros and Cons

| Pros | Cons |
|------|------|
| Easy to understand and visualize | Highly prone to overfitting if not pruned |
| Requires almost no data preparation (No scaling needed!) | **High Variance**: A tiny change in training data can result in a completely different tree |
| Automatically performs feature selection | Biased toward dominant classes (if dataset is imbalanced) |

**Next step:** How do we fix the High Variance problem? By planting a forest (Random Forests)!